# Edges in LangGraph
Companion notebook for the **Edges in LangGraph** section of Chapter 4.

Static edges always move execution the same way; conditional edges pick the next node at runtime based on a routing function.

In [1]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END


class ResearchState(BaseModel):
    query: str = ""
    candidate_urls: list[str] = Field(default_factory=list)
    page_texts: list[str] = Field(default_factory=list)
    draft_summary: str = ""
    final_report: str = ""
    errors: list[str] = Field(default_factory=list)

The chapter references a few helper functions (`normalize_query`, `enrich_data`, ...) without spelling them out. Simple stand-ins below keep the example runnable.

In [2]:
def normalize_query(query: str) -> str:
    return query.strip().lower()


def initial_text_processing(text: str) -> str:
    return f"[processed] {text}"


def enrich_data(text: str) -> str:
    return f"{text} + context"


def semantic_analysis(text: str) -> str:
    return f"{text} + semantics"


def format_report(text: str) -> str:
    return f"[formatted] {text}"


def compile_final_report(text: str) -> str:
    return f"Final Report -> {text}"

In [3]:
def node_a(state: ResearchState) -> ResearchState:
    """Node A - Query Validator & Initial Processor"""
    normalized = normalize_query(state.query)
    processed = initial_text_processing(normalized)
    state.draft_summary = processed
    return state


def node_b(state: ResearchState) -> ResearchState:
    """Node B - Data Enrichment & Enhancement"""
    enriched = enrich_data(state.draft_summary)
    analyzed = semantic_analysis(enriched)
    state.draft_summary = analyzed
    return state


def node_c(state: ResearchState) -> ResearchState:
    """Node C - Final Report Compilation"""
    formatted = format_report(state.draft_summary)
    compiled = compile_final_report(formatted)
    state.final_report = compiled
    return state


def route_based_on_query(state: ResearchState) -> str:
    if len(state.query) > 20:
        return "node_c"
    else:
        return END

In [4]:
workflow = StateGraph(ResearchState)
workflow.add_node("node_a", node_a)
workflow.add_node("node_b", node_b)
workflow.add_node("node_c", node_c)

workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_conditional_edges(
    "node_b",
    route_based_on_query,
    {
        "node_c": "node_c",
        END: END,
    },
)

graph = workflow.compile()

In [5]:
short_result = graph.invoke(ResearchState(query="short one"))
print("Short query (<=20 chars) -> stops after node_b")
print("draft_summary:", short_result["draft_summary"])
print("final_report:", repr(short_result["final_report"]))

print()

long_result = graph.invoke(ResearchState(query="Explain recent trends in electric vehicles"))
print("Long query (>20 chars) -> continues to node_c")
print("draft_summary:", long_result["draft_summary"])
print("final_report:", long_result["final_report"])

Short query (<=20 chars) -> stops after node_b
draft_summary: [processed] short one + context + semantics
final_report: ''

Long query (>20 chars) -> continues to node_c
draft_summary: [processed] explain recent trends in electric vehicles + context + semantics
final_report: Final Report -> [formatted] [processed] explain recent trends in electric vehicles + context + semantics
